In [4]:
import numpy as np
from scipy.optimize import minimize, differential_evolution
import warnings
warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — DATA KOMPONEN MURNI (INPUT)
# ══════════════════════════════════════════════════════════════════════════════
# Isi nilai κAB dan εAB/k (K) untuk setiap molekul yang digunakan fitting.
# Nama kunci bebas, tapi harus SAMA dengan yang dipakai di SECTION 4 & 6.
# Tambah atau hapus baris sesuai kebutuhan.
#
# Format: "nama_molekul" : (kappa_AB, epsilon_AB_per_k_Kelvin)
# PERHATIAN: urutan di dalam kurung adalah (κAB, εAB) — konsisten!

pure_components = {
    #   nama                  κAB              εAB/K
    "acetic"      : (0.413948641,   2375.020391),
    "propionic"   : (0.125450989,   2743.424723),
    "butyric"     : (0.068716339,   2761.69866),
    "valeric"     : (0.048068268,   2475.378108),
    "isobutyric"   : (0.002050858,  2549.064822),
    "isovaleric"    : (0.009567705, 2601.89495),
    "isocaproic"   : (0.022218218,  2445.491986)
    # ← tambahkan molekul lain di sini jika perlu, jangan lupa koma
    # "butylamine"       : (nilai_kappa,   nilai_epsilon),
}

# Urutan molekul — harus SAMA dengan urutan kolom di S_terms (SECTION 4)
molecules = ["acetic", "propionic", "butyric", "valeric", "isobutyric", "isovaleric", "isocaproic"]


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 2 — PARAMETER REFERENSI GRUP (INPUT — SUDAH DIKETAHUI)
# ══════════════════════════════════════════════════════════════════════════════
# Nilai κAB_α dan εAB_α untuk setiap grup referensi.
# Nilai ini TIDAK di-fitting — sudah ditentukan sebelumnya.
# Grup yang belum diketahui nilainya → jadikan unknown di SECTION 5
#   (contoh: jika κAB_NH2 belum diketahui, buat unknown "NH2-ref-kap")
#
# Grup yang tidak punya situs asosiasi (CH3, CH2, CH):
#   Secara fisik κAB = 0 dan εAB = 0, tapi dalam persamaan dokumen
#   ini tetap muncul sebagai n_α × κAB_α — isi 0.0 jika memang nol.
#
# Format: "GRUP" : (kappa_AB_ref, epsilon_AB_ref_per_K)

GC_ref = {
    #  grup     κAB_ref      εAB_ref/K
    "CH3"  : (0.0,          0.0      ),   # tidak asosiatif
    "CH2"  : (0.0,          0.0      ),   # tidak asosiatif
    "CH"   : (0.0,          0.0      ),   # tidak asosiatif
    "NH2"  : (0.149350,     504.320  ),   # dari Vijande et al. Table 1
    "OH"   : (0.001341,     2217.87  ),   # dari Vijande et al. Table 1
    #"CONH" : (0.0,          0.0      ),   # ← isi jika ada dan diketahui
    # ← tambahkan grup baru di sini jika perlu, jangan lupa koma
}


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — PARAMETER PERTURBATIF μ YANG SUDAH DIKETAHUI (INPUT)
# ══════════════════════════════════════════════════════════════════════════════
# Nilai μ_{κAB} dan μ_{εAB} untuk pasangan yang sudah diketahui.
# Pasangan yang BELUM diketahui → masuk ke SECTION 5 sebagai unknown.
#
# CATATAN: untuk parameter asosiasi, tidak ada konversi seperti μ(mε).
#   Nilai langsung dipakai dalam persamaan tanpa perlu dikalikan faktor apapun.
#
# Format: "GRUP1-GRUP2" : {"kap": nilai_mu_kappa, "eps": nilai_mu_epsilon}

mu_known = {
    # Isi nilai μ_{κAB} dan μ_{εAB} untuk pasangan yang SUDAH DIKETAHUI.
    # Jika pasangan ada di persamaan (SECTION 6) tapi nilainya belum diketahui
    # → pindahkan ke unknown_pairs di SECTION 5.
    # Jika nilai interaksi = 0, tetap tulis di sini dengan nilai 0.0
    #
    "CH3-CH3"  : {"kap":  0.0,   "eps":  0.0  },   # ← ganti dengan nilai sebenarnya
    "CH3-CH"   : {"kap":  0.0,   "eps":  0.0  },   # ← ganti dengan nilai sebenarnya
    "CH2-CH3"  : {"kap":  0.0,   "eps":  0.0  },   # ← ganti dengan nilai sebenarnya
    "CH2-CH2"  : {"kap":  0.0,   "eps":  0.0  },
    "CH2-CH"   : {"kap":  0.0,   "eps":  0.0  },   # ← ganti dengan nilai sebenarnya
    "CH3-NH2"  : {"kap":  -0.0311,   "eps":  111.93  },   # ← ganti dengan nilai sebenarnya
    "CH2-NH2"  : {"kap":  0.0,   "eps":  0.0  },   # ← ganti dengan nilai sebenarnya
    "CH3-OH"   : {"kap":  0.0484919,   "eps":  619.549},
    "CH2-OH"   : {"kap":  0.0,   "eps":  0.0  },
    "CH2-COOH"  :{"kap":  0.0,   "eps":  0.0  }
    # ← tambahkan pasangan known lainnya di sini, jangan lupa koma
}


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 4 — S-TERM (INPUT MANUAL)
# ══════════════════════════════════════════════════════════════════════════════
# S-term untuk setiap pasangan grup, untuk setiap molekul.
# Urutan kolom harus SAMA dengan urutan di variabel 'molecules' (SECTION 1).
#
# Cara menghitung S_{α-β} (persamaan 2 Vijande et al.):
#   S_{α-β} = Σ_i Σ_j  1 / p_{αi-βj}
#   p_{αi-βj} = jumlah bond yang memisahkan grup αi dan βj.
#   Pasangan yang sama (i=j) tidak dihitung.
#
# Jika pasangan tidak ada dalam molekul, isikan 0.0
#
# Format: "GRUP1-GRUP2" : [nilai_mol1, nilai_mol2, ...]

S_terms = {
    # urutan kolom = urutan 'molecules': [isopropylamine, secbutylamine]
    #
    #                 
   "CH3-CH3"    : [ 0, 0, 0, 0, 0.5, 0.5, 0.5],
    "CH3-CH"     : [ 0, 0, 0, 0, 2, 2, 2],
   "CH2-CH3"    : [  0, 1, 1.5, 1.833333333, 0, 1, 1.666666667],
    "CH2-CH2"    : [ 0, 0, 0, 0,  0, 0, 1],
    "CH2-CH"     : [ 0, 0, 0, 0,  0, 1, 1.5],
    "CH3-COOH"    : [ 1.0, 0.5,  0.333333333, 0.25, 1, 0.666666667, 0.5],
    "CH2-COOH"    : [ 0.0, 1.0,  1.5, 1.833333333, 0, 1, 1.5],
    "CH-COOH"     : [ 0, 0, 0, 0,  1, 0.5, 0.333333333]
    # ← tambahkan S-term pasangan lain jika dibutuhkan di SECTION 6
    # jangan lupa koma di setiap baris kecuali baris terakhir
}


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 5 — VARIABEL YANG DI-FITTING (UNKNOWN)
# ══════════════════════════════════════════════════════════════════════════════
# Daftar parameter yang BELUM diketahui dan akan di-fitting.
# Nama bebas tapi harus konsisten dengan yang dipakai di SECTION 6.
#
# Untuk setiap properti (kap=κAB, eps=εAB), ada 1 nilai per unknown.
# Total parameter di bounds = len(unknown_pairs) × 2
#
# Contoh jenis unknown yang bisa dimasukkan:
#   "CH-NH2"      → μ_{κAB, CH-NH2} dan μ_{εAB, CH-NH2}
#   "NH2-ref"     → κAB_NH2 dan εAB_NH2 referensi (jika belum diketahui)
#   "CH3-NH2"     → μ_{κAB, CH3-NH2} dan μ_{εAB, CH3-NH2}

unknown_pairs = [
    "COOH-ref",
    "CH3-COOH",
    "CH-COOH"
    # ← tambahkan unknown lain di sini, jangan lupa koma
    # "CH3-NH2",
    # "NH2-ref",
]

# Jumlah unknown per properti (dihitung otomatis)
N_unk = len(unknown_pairs)


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 6 — PERSAMAAN MODEL PER MOLEKUL
# ══════════════════════════════════════════════════════════════════════════════
# Di sinilah Anda menulis persamaan κAB_GC dan εAB_GC untuk tiap molekul.
#
# CARA PENGGUNAAN:
#   gc(grup, prop)          → nilai referensi grup κAB atau εAB (SECTION 2)
#   mk(pasangan, prop)      → nilai μ known (SECTION 3)
#   S(pasangan, mol_idx)    → nilai S-term (SECTION 4)
#   unk[i]                  → nilai unknown ke-i (urutan = SECTION 5)
#
# prop bisa berisi: "kap" (untuk κAB) atau "eps" (untuk εAB)
#
# KOEFISIEN (1 atau 2 di depan μ·S):
#   2× jika pasangan SIMETRIS (contoh: 2 CH3 yang keduanya berinteraksi dengan NH2)
#   1× jika tidak simetris atau hanya ada 1 pasang grup tersebut
#
# SUKU REFERENSI:
#   Untuk asam amino / amina: SEMUA grup ditulis n_α × gc(grup, prop)
#   meskipun CH3/CH2/CH nilainya 0 (tidak asosiatif) — tetap tulis untuk kejelasan
#
# mol_idx: indeks molekul sesuai urutan 'molecules' (0 = molekul pertama, dst.)

def model(mol_idx, prop, unk):
    """
    Hitung κAB_GC atau εAB_GC untuk molekul mol_idx.

    Parameter
    ---------
    mol_idx : int   → indeks molekul (0 = isopropylamine, 1 = secbutylamine)
    prop    : str   → "kap" untuk κAB, atau "eps" untuk εAB
    unk     : list  → nilai unknown saat ini, urutan sesuai unknown_pairs

    Kembalikan
    ----------
    float → nilai κAB_GC atau εAB_GC
    """

    # ── Ambil nilai unknown dengan nama yang jelas ──────────────────────────
    # Sesuaikan dengan urutan di unknown_pairs (SECTION 5)
    COOH_ref = unk[0]
    mu_CH3_COOH = unk[1]
    mu_CH_COOH = unk[2]

    # ── Persamaan per molekul ────────────────────────────────────────────────

    if mol_idx == 0:
        #acetic
        return (
            1 * gc("CH3", prop) 
            + 0 * gc("CH2", prop)
            + 0 * gc("CH", prop)
            + 1 * COOH_ref
            + 1 * mk("CH3-CH3", prop) * S("CH3-CH3", mol_idx)
            + 1 * mk("CH3-CH", prop) * S("CH3-CH", mol_idx)
            + 1 * mk("CH2-CH3", prop) * S("CH2-CH3", mol_idx)
            + 1 * mk("CH2-CH2", prop) * S("CH2-CH2", mol_idx)
            + 1 * mk("CH2-CH", prop) * S("CH2-CH", mol_idx)
            + 1 * mu_CH3_COOH * S("CH3-COOH", mol_idx)
            + 1 *  mk("CH2-COOH", prop) * S("CH2-COOH", mol_idx)
            + 1 * mu_CH_COOH * S("CH-COOH", mol_idx)
        )

    elif mol_idx == 1:
        #propionic
        return (
            1 * gc("CH3", prop) 
            + 1 * gc("CH2", prop)
            + 0 * gc("CH", prop)
            + 1 * COOH_ref
            + 1 * mk("CH3-CH3", prop) * S("CH3-CH3", mol_idx)
            + 1 * mk("CH3-CH", prop) * S("CH3-CH", mol_idx)
            + 1 * mk("CH2-CH3", prop) * S("CH2-CH3", mol_idx)
            + 1 * mk("CH2-CH2", prop) * S("CH2-CH2", mol_idx)
            + 1 * mk("CH2-CH", prop) * S("CH2-CH", mol_idx)
            + 1 * mu_CH3_COOH * S("CH3-COOH", mol_idx)
            + 1 *  mk("CH2-COOH", prop) * S("CH2-COOH", mol_idx)
            + 1 * mu_CH_COOH * S("CH-COOH", mol_idx)
        )
    elif mol_idx == 2:
        #BUTYRIC
        return (
            1 * gc("CH3", prop) 
            + 2 * gc("CH2", prop)
            + 0 * gc("CH", prop)
            + 1 * COOH_ref
            + 1 * mk("CH3-CH3", prop) * S("CH3-CH3", mol_idx)
            + 1 * mk("CH3-CH", prop) * S("CH3-CH", mol_idx)
            + 1 * mk("CH2-CH3", prop) * S("CH2-CH3", mol_idx)
            + 1 * mk("CH2-CH2", prop) * S("CH2-CH2", mol_idx)
            + 1 * mk("CH2-CH", prop) * S("CH2-CH", mol_idx)
            + 1 * mu_CH3_COOH * S("CH3-COOH", mol_idx)
            + 1 *  mk("CH2-COOH", prop) * S("CH2-COOH", mol_idx)
            + 1 * mu_CH_COOH * S("CH-COOH", mol_idx)
        )

    elif mol_idx == 3:
        #valeric
        return (
            1 * gc("CH3", prop) 
            + 3 * gc("CH2", prop)
            + 0 * gc("CH", prop)
            + 1 * COOH_ref
            + 1 * mk("CH3-CH3", prop) * S("CH3-CH3", mol_idx)
            + 1 * mk("CH3-CH", prop) * S("CH3-CH", mol_idx)
            + 1 * mk("CH2-CH3", prop) * S("CH2-CH3", mol_idx)
            + 1 * mk("CH2-CH2", prop) * S("CH2-CH2", mol_idx)
            + 1 * mk("CH2-CH", prop) * S("CH2-CH", mol_idx)
            + 1 * mu_CH3_COOH * S("CH3-COOH", mol_idx)
            + 1 *  mk("CH2-COOH", prop) * S("CH2-COOH", mol_idx)
            + 1 * mu_CH_COOH * S("CH-COOH", mol_idx)
        )

    elif mol_idx == 4:
        #isobutyr
        return (
            2 * gc("CH3", prop) 
            + 0 * gc("CH2", prop)
            + 1 * gc("CH", prop)
            + 1 * COOH_ref
            + 1 * mk("CH3-CH3", prop) * S("CH3-CH3", mol_idx)
            + 1 * mk("CH3-CH", prop) * S("CH3-CH", mol_idx)
            + 1 * mk("CH2-CH3", prop) * S("CH2-CH3", mol_idx)
            + 1 * mk("CH2-CH2", prop) * S("CH2-CH2", mol_idx)
            + 1 * mk("CH2-CH", prop) * S("CH2-CH", mol_idx)
            + 1 * mu_CH3_COOH * S("CH3-COOH", mol_idx)
            + 1 *  mk("CH2-COOH", prop) * S("CH2-COOH", mol_idx)
            + 1 * mu_CH_COOH * S("CH-COOH", mol_idx)
        )

    elif mol_idx == 5:
        #isovaleric
        return (
            2 * gc("CH3", prop) 
            + 1 * gc("CH2", prop)
            + 1 * gc("CH", prop)
            + 1 * COOH_ref
            + 1 * mk("CH3-CH3", prop) * S("CH3-CH3", mol_idx)
            + 1 * mk("CH3-CH", prop) * S("CH3-CH", mol_idx)
            + 1 * mk("CH2-CH3", prop) * S("CH2-CH3", mol_idx)
            + 1 * mk("CH2-CH2", prop) * S("CH2-CH2", mol_idx)
            + 1 * mk("CH2-CH", prop) * S("CH2-CH", mol_idx)
            + 1 * mu_CH3_COOH * S("CH3-COOH", mol_idx)
            + 1 *  mk("CH2-COOH", prop) * S("CH2-COOH", mol_idx)
            + 1 * mu_CH_COOH * S("CH-COOH", mol_idx)
        )

    elif mol_idx == 6:
        #isocapro
        return (
            2 * gc("CH3", prop) 
            + 2 * gc("CH2", prop)
            + 1 * gc("CH", prop)
            + 1 * COOH_ref
            + 1 * mk("CH3-CH3", prop) * S("CH3-CH3", mol_idx)
            + 1 * mk("CH3-CH", prop) * S("CH3-CH", mol_idx)
            + 1 * mk("CH2-CH3", prop) * S("CH2-CH3", mol_idx)
            + 1 * mk("CH2-CH2", prop) * S("CH2-CH2", mol_idx)
            + 1 * mk("CH2-CH", prop) * S("CH2-CH", mol_idx)
            + 1 * mu_CH3_COOH * S("CH3-COOH", mol_idx)
            + 1 *  mk("CH2-COOH", prop) * S("CH2-COOH", mol_idx)
            + 1 * mu_CH_COOH * S("CH-COOH", mol_idx)
        )

    

    else:
        raise ValueError(f"mol_idx={mol_idx} tidak ada dalam daftar molekul.")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 7 — BATAS (BOUNDS) UNTUK UNKNOWN
# ══════════════════════════════════════════════════════════════════════════════
# Urutan: untuk κAB dulu, lalu εAB — masing-masing N_unk baris.
# Total batas = N_unk × 2
#
# Format: (batas_bawah, batas_atas) — harus dalam bentuk float (angka desimal)
#
# Panduan fisik:
#   κAB referensi grup  : 0.0 – 1.0  (tidak bisa negatif secara fisik)
#   εAB referensi grup  : 0 – 5000 K
#   μ_{κAB}             : ±1.0  (perturbasi relatif kecil)
#   μ_{εAB}             : ±2000 K
#
# Jika setelah running ada parameter yang menempel di batas → perlebar batas.

bounds = [
   # kapa
    (-10, 10), (-1.0, 1.0), (-1.0, 1.0),

    # epsilon
    (100, 10000), (-3000.0, 3000.0), (-3000.0, 3000.0)
]

# Validasi otomatis — tidak perlu diedit
assert len(bounds) == N_unk * 2, (
    f"Jumlah bounds ({len(bounds)}) harus = N_unk×2 = {N_unk*2}. "
    f"Sesuaikan SECTION 7: {N_unk} baris untuk κAB, lalu {N_unk} baris untuk εAB."
)


# ══════════════════════════════════════════════════════════════════════════════
# BAGIAN OTOMATIS — TIDAK PERLU DIEDIT
# (Helper functions, objective function, optimizer, output)
# ══════════════════════════════════════════════════════════════════════════════

# ── Helper: ambil nilai referensi grup ──────────────────────────────────────
def gc(group, prop):
    """
    Kembalikan nilai referensi κAB atau εAB untuk grup.
    prop = "kap" → κAB_grup
    prop = "eps" → εAB_grup
    """
    kap_g, eps_g = GC_ref[group]
    return {"kap": kap_g, "eps": eps_g}[prop]

# ── Helper: ambil μ known ────────────────────────────────────────────────────
def mk(pair, prop):
    """Kembalikan nilai μ yang sudah diketahui untuk pasangan dan properti."""
    if pair not in mu_known:
        raise KeyError(
            f"Pasangan '{pair}' tidak ada di mu_known (SECTION 3). "
            f"Tambahkan jika diketahui, atau masukkan ke unknown_pairs (SECTION 5)."
        )
    return mu_known[pair][prop]

# ── Helper: ambil S-term ─────────────────────────────────────────────────────
def S(pair, mol_idx):
    """Kembalikan nilai S-term untuk pasangan dan indeks molekul."""
    if pair not in S_terms:
        raise KeyError(
            f"Pasangan '{pair}' tidak ada di S_terms (SECTION 4). "
            f"Tambahkan nilainya."
        )
    return S_terms[pair][mol_idx]

# ── Hitung target dari data komponen murni ───────────────────────────────────
# targets["kap"] = array κAB tiap molekul
# targets["eps"] = array εAB tiap molekul
targets = {"kap": [], "eps": []}
for mol in molecules:
    kap, eps = pure_components[mol]
    targets["kap"].append(kap)
    targets["eps"].append(eps)
targets = {prop: np.array(v) for prop, v in targets.items()}

# ── Unpack unknown dari vektor x ─────────────────────────────────────────────
def unpack(x, prop):
    """
    Ambil slice unknown untuk properti tertentu dari vektor x optimizer.
    Urutan dalam x: [kap_unk0, kap_unk1, ..., eps_unk0, eps_unk1, ...]
    """
    i = {"kap": 0, "eps": 1}[prop]
    return x[i * N_unk : (i + 1) * N_unk]

# ── Objective function ────────────────────────────────────────────────────────
def objective(x):
    """
    F_OBJ = Σ_prop Σ_molekul [ (π_target - π_GC) / π_target ]²

    prop ∈ {kap (κAB), eps (εAB)}

    Vektor x:
      x[0 : N_unk]       → unknown untuk κAB
      x[N_unk : 2*N_unk] → unknown untuk εAB
    """
    F = 0.0
    for prop in ("kap", "eps"):
        unk = unpack(x, prop)
        for mol_idx in range(len(molecules)):
            pi_GC  = model(mol_idx, prop, unk)
            pi_tgt = targets[prop][mol_idx]
            # Hindari pembagian dengan nol jika target = 0
            if abs(pi_tgt) > 1e-15:
                F += ((pi_tgt - pi_GC) / pi_tgt) ** 2
    return F

# ── Optimasi dua tahap: global → lokal ───────────────────────────────────────
print("=" * 62)
print("  PC-SAFT PertGC — Association Parameter Fitting Template")
print("=" * 62)
print(f"\n  Molekul  : {', '.join(molecules)}")
print(f"  Unknown  : {', '.join(unknown_pairs)}")
print(f"  Total    : {N_unk} unknown × 2 properti = {N_unk*2} parameter\n")

print("  Stage 1: Global search (differential_evolution)...")
res_g = differential_evolution(
    objective, bounds,
    seed=42, maxiter=8000, tol=1e-14,
    popsize=25, mutation=(0.5, 1.5), recombination=0.9,
    polish=True, disp=False,
)

print("  Stage 2: Local refinement (L-BFGS-B)...")
res_l = minimize(
    objective, res_g.x,
    method="L-BFGS-B", bounds=bounds,
    options={"maxiter": 100000, "ftol": 1e-16, "gtol": 1e-13},
)

x_opt = res_l.x if res_l.fun < res_g.fun else res_g.x
F_opt = min(res_l.fun, res_g.fun)

# ── Cetak parameter hasil fitting ────────────────────────────────────────────
prop_label = {"kap": "κAB", "eps": "εAB (K)"}

print("\n" + "=" * 62)
print("  FITTED PARAMETERS")
print("=" * 62)
for prop in ("kap", "eps"):
    unk = unpack(x_opt, prop)
    print(f"\n  Properti: {prop_label[prop]}")
    for j, pair in enumerate(unknown_pairs):
        print(f"    μ_{prop_label[prop].split()[0]}({pair:<18}) = {unk[j]:>16.6f}")

print(f"\n  F_OBJ (final) = {F_opt:.6e}")

# ── Validasi: target vs. prediksi GC ─────────────────────────────────────────
print("\n" + "=" * 62)
print("  VALIDATION — Target vs. GC-Predicted (APD%)")
print("=" * 62)

for prop in ("kap", "eps"):
    unk = unpack(x_opt, prop)
    print(f"\n  {prop_label[prop]}")
    print(f"  {'Molekul':<20} {'Target':>14} {'GC-Pred':>14} {'APD%':>9}")
    print(f"  {'-'*60}")
    for mol_idx, mol in enumerate(molecules):
        pi_tgt = targets[prop][mol_idx]
        pi_gc  = model(mol_idx, prop, unk)
        apd    = 100.0 * abs(pi_tgt - pi_gc) / abs(pi_tgt) if abs(pi_tgt) > 1e-15 else 0.0
        print(f"  {mol:<20} {pi_tgt:>14.6f} {pi_gc:>14.6f} {apd:>8.4f}%")

# ── Tabel ringkasan ───────────────────────────────────────────────────────────
print("\n" + "=" * 62)
print("  SUMMARY TABLE (copy-paste ready)")
print("=" * 62)
print(f"  {'Parameter':<28} {'κAB':>14} {'εAB (K)':>14}")
print("  " + "-" * 58)
for j, pair in enumerate(unknown_pairs):
    val_kap = unpack(x_opt, "kap")[j]
    val_eps = unpack(x_opt, "eps")[j]
    print(f"  μ({pair:<24}) {val_kap:>14.6f} {val_eps:>14.4f}")

# ── Diagnostik batas ──────────────────────────────────────────────────────────
print("\n" + "=" * 62)
print("  DIAGNOSTIC — Bound Check")
print("=" * 62)

all_labels = (
    [f"μ_kap({p})" for p in unknown_pairs] +
    [f"μ_eps({p})" for p in unknown_pairs]
)

bound_hit = False
for i, (val, (lo, hi)) in enumerate(zip(x_opt, bounds)):
    tol = max(1e-4, 1e-4 * abs(hi - lo))
    if abs(val - lo) < tol or abs(val - hi) < tol:
        print(f"  ⚠  {all_labels[i]} = {val:.6f} menempel di batas [{lo}, {hi}].")
        print(f"      → Perlebar batas ini di SECTION 7 dan jalankan ulang.")
        bound_hit = True
if not bound_hit:
    print("  ✓  Tidak ada parameter yang menempel di batas. Solusi interior.")

  PC-SAFT PertGC — Association Parameter Fitting Template

  Molekul  : acetic, propionic, butyric, valeric, isobutyric, isovaleric, isocaproic
  Unknown  : COOH-ref, CH3-COOH, CH-COOH
  Total    : 3 unknown × 2 properti = 6 parameter

  Stage 1: Global search (differential_evolution)...
  Stage 2: Local refinement (L-BFGS-B)...

  FITTED PARAMETERS

  Properti: κAB
    μ_κAB(COOH-ref          ) =        -0.012292
    μ_κAB(CH3-COOH          ) =         0.129695
    μ_κAB(CH-COOH           ) =        -0.115487

  Properti: εAB (K)
    μ_εAB(COOH-ref          ) =      2676.778915
    μ_εAB(CH3-COOH          ) =      -239.617839
    μ_εAB(CH-COOH           ) =        82.499712

  F_OBJ (final) = 2.159874e+00

  VALIDATION — Target vs. GC-Predicted (APD%)

  κAB
  Molekul                      Target        GC-Pred      APD%
  ------------------------------------------------------------
  acetic                     0.413949       0.117403  71.6383%
  propionic                  0.125451    